In [6]:
import pandas as pd

sentence_level_df = pd.read_csv("alias_sentence_features_primary_sorted.csv")

In [7]:
sentence_level_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5983 entries, 0 to 5982
Data columns (total 12 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   story_id                5983 non-null   int64  
 1   person                  5983 non-null   object 
 2   aliases                 5983 non-null   object 
 3   type                    5983 non-null   object 
 4   sentence_id             5983 non-null   int64  
 5   text                    5980 non-null   object 
 6   mention_count           5983 non-null   int64  
 7   word_count              5983 non-null   float64
 8   text_prev               5799 non-null   object 
 9   text_next               5853 non-null   object 
 10  bert_context            5980 non-null   object 
 11  is_primary_in_sentence  5983 non-null   int64  
dtypes: float64(1), int64(4), object(7)
memory usage: 561.0+ KB


In [8]:
sentence_level_df = sentence_level_df.drop(columns=[
    "mention_count",
    "word_count",
    "text_prev",
    "text_next",
    "bert_context",
    "is_primary_in_sentence"
])

In [9]:
import pandas as pd

def is_dialogue(sentence):
    if isinstance(sentence, str):
        return int(any(q in sentence for q in ['"', '“', '”', '‘', '’']))
    return 0

# Apply to your DataFrame
sentence_level_df['is_dialogue'] = sentence_level_df['text'].apply(is_dialogue)
sentence_level_df

,story_id,person,aliases,type,sentence_id,text,is_dialogue
0,1,Tokoh-1,"['Pan Karsa', 'Pan Karta']",Antagonist,1,Pan Karsa ajaka pianakne muani nanggap upah ng...,0
1,1,Tokoh-1,"['Pan Karsa', 'Pan Karta']",Antagonist,9,Apeteng Pan Karsa tusing bisa pules ngenehang...,0
2,1,Tokoh-1,"['Pan Karsa', 'Pan Karta']",Antagonist,14,Pan Karsa masih mapangenan,0
3,1,Tokoh-1,"['Pan Karsa', 'Pan Karta']",Antagonist,19,Ditu lantas Pan Karsa ngantungan baju muah ca...,0
4,1,Tokoh-1,"['Pan Karsa', 'Pan Karta']",Antagonist,29,Ditu Pan Karta malaib-laib tur ngomong Inggih...,0
...,...,...,...,...,...,...,...
5978,120,Tokoh-11,"['I Udang', 'udange']",Protagonist,110,Ngedikin ia I Yuyu kanti menek taine ketendas...,0
5979,120,Tokoh-12,"['kebone', 'kebone gede']",Protagonist,106,Ditu kone kebone celepanga kesongne I Yuyu na...,0
5980,120,Tokoh-12,"['kebone', 'kebone gede']",Protagonist,107,Sangkane kayang jani tundun yuyune misi enjek...,0
5981,120,Tokoh-13,"['adi kekua', 'adi']",Protagonist,49,adi kekua iraga suba matimpal uli nguni jele ...,0


In [10]:
import pandas as pd

# ── 2. LOAD AFINN TSV ───────────────────────────────────
def load_lex(lex_path):
    # File AFINN.tsv kamu punya header: term, valence, score
    lex_df = pd.read_excel(lex_path)

    # Pastikan term lowercase agar cocok dengan lemma.lower()
    lex_df["word"] = lex_df["word"].astype(str).str.lower()
    lex_df["score"] = lex_df["score"].astype(float)

    # Jadikan dictionary: kata -> skor
    lexicon = dict(zip(lex_df["word"], lex_df["score"]))

    return lexicon

# ── 3. FUNGSI HITUNG SKOR LEXICON VERB + ADJ ────────────
def sentiment_lexicon(text, lexicon):
    if pd.isna(text):
        return 0

    text = str(text).strip().lower()
    if text == "":
        return 0

    score_total = 0

    # Tokenisasi sederhana berdasarkan spasi
    words = text.split()

    for token in words:
        if token in lexicon:
            score_total += lexicon[token]

    return score_total

# ── 4. LOAD LEXICON ─────────────────────────────────────
lex_path = "lexicon_bali.xlsx"
lexicon = load_lex(lex_path)

# Cek sebagian isi lexicon
print("Jumlah kata dalam lexicon:", len(lexicon))
print(list(lexicon.items())[:10])

# ── 5. TERAPKAN KE DATASET ──────────────────────────────
sentence_level_df["count_lexicon"] = sentence_level_df["text"].apply(
    lambda x: sentiment_lexicon(x, lexicon)
)

# Cek hasil
sentence_level_df[["text", "count_lexicon"]].head()

Jumlah kata dalam lexicon: 164
[('gede', 2.0), ('bales', 2.0), ('sedih', -5.0), ('mentas', 0.0), ('mara', 0.0), ('liu', 1.0), ('kedas', 4.0), ('melah', 5.0), ('jemet', 4.0), ('wicaksana', 5.0)]


,text,count_lexicon
0,Pan Karsa ajaka pianakne muani nanggap upah ng...,2.0
1,Apeteng Pan Karsa tusing bisa pules ngenehang...,0.0
2,Pan Karsa masih mapangenan,0.0
3,Ditu lantas Pan Karsa ngantungan baju muah ca...,0.0
4,Ditu Pan Karta malaib-laib tur ngomong Inggih...,1.0


In [11]:
import pandas as pd
import ast
import re

# =========================================================
# 1. FUNGSI AMAN UNTUK PARSE ALIASES
# =========================================================
def safe_parse_aliases(x):
    if isinstance(x, list):
        return x

    if pd.isna(x):
        return []

    if isinstance(x, str):
        x = x.strip()

        if not x:
            return []

        try:
            parsed = ast.literal_eval(x)

            if isinstance(parsed, list):
                return parsed

            return [str(parsed)]

        except:
            return [x]

    return []


# =========================================================
# 2. NORMALISASI TEKS
# =========================================================
def normalize_text(text):
    text = str(text).lower().strip()
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text


# =========================================================
# 3. TOKENISASI SEDERHANA
# =========================================================
def simple_tokenize(text):
    text = normalize_text(text)
    return text.split()


# =========================================================
# 4. CARI POSISI ALIAS DALAM TOKEN KALIMAT
# =========================================================
def find_alias_position(sentence_tokens, alias_tokens):
    alias_len = len(alias_tokens)

    for i in range(len(sentence_tokens) - alias_len + 1):
        window = sentence_tokens[i:i + alias_len]

        if window == alias_tokens:
            return i, i + alias_len - 1

    return None


# =========================================================
# 5. FUNGSI SUBJECT / OBJECT TANPA NLP DAN TANPA MARKERS
# =========================================================
def get_alias_roles_without_nlp(row):
    sentence = row["text"]
    aliases = safe_parse_aliases(row["aliases"])

    if pd.isna(sentence):
        return pd.Series({
            "is_subject": 0,
            "is_object": 0
        })

    sentence = str(sentence).strip()

    if sentence == "":
        return pd.Series({
            "is_subject": 0,
            "is_object": 0
        })

    sentence_tokens = simple_tokenize(sentence)

    alias_positions = []

    for alias in aliases:
        alias = normalize_text(alias)

        if alias == "":
            continue

        alias_tokens = alias.split()

        position = find_alias_position(sentence_tokens, alias_tokens)

        if position is not None:
            start_idx, end_idx = position

            alias_positions.append({
                "alias": alias,
                "start_idx": start_idx,
                "end_idx": end_idx
            })

    # Kalau tidak ada alias yang muncul di kalimat
    if len(alias_positions) == 0:
        return pd.Series({
            "is_subject": 0,
            "is_object": 0
        })

    # Urutkan berdasarkan posisi kemunculan alias dalam kalimat
    alias_positions = sorted(alias_positions, key=lambda x: x["start_idx"])

    is_subject = 0
    is_object = 0

    # Alias pertama dianggap subject
    if len(alias_positions) >= 1:
        is_subject = 1

    # Alias kedua dan seterusnya dianggap object
    if len(alias_positions) >= 2:
        is_object = 1

    return pd.Series({
        "is_subject": is_subject,
        "is_object": is_object
    })


# =========================================================
# 6. TERAPKAN KE DATASET
# =========================================================
sentence_level_df[["is_subject", "is_object"]] = sentence_level_df.apply(
    get_alias_roles_without_nlp,
    axis=1
)

# =========================================================
# 7. CEK HASIL
# =========================================================
sentence_level_df[["text", "aliases", "is_subject", "is_object"]].head(5)

,text,aliases,is_subject,is_object
0,Pan Karsa ajaka pianakne muani nanggap upah ng...,"['Pan Karsa', 'Pan Karta']",1,0
1,Apeteng Pan Karsa tusing bisa pules ngenehang...,"['Pan Karsa', 'Pan Karta']",1,0
2,Pan Karsa masih mapangenan,"['Pan Karsa', 'Pan Karta']",1,0
3,Ditu lantas Pan Karsa ngantungan baju muah ca...,"['Pan Karsa', 'Pan Karta']",1,0
4,Ditu Pan Karta malaib-laib tur ngomong Inggih...,"['Pan Karsa', 'Pan Karta']",1,0


In [12]:
sentence_level_df = sentence_level_df.dropna(subset=["text"]).reset_index(drop=True)

In [13]:
print(sentence_level_df.columns)

Index(['story_id', 'person', 'aliases', 'type', 'sentence_id', 'text',
       'is_dialogue', 'count_lexicon', 'is_subject', 'is_object'],
      dtype='object')


In [14]:
sentence_level_df.to_csv("data_bali_ferro_base.csv", index=False)

In [15]:
import pandas as pd

# Step 1: Identify the least frequent label
label_counts = sentence_level_df['type'].value_counts()
least_common_label = label_counts.idxmin()

# Step 2: Sort so that the least common label comes first (we want to keep these)
sentence_level_df_sorted = sentence_level_df.sort_values(
    by='type',
    key=lambda x: x.apply(lambda val: 0 if val == least_common_label else 1)
)

# Step 3: Drop duplicates in 'subject_sentences' keeping the first occurrence (which is from least_common_label)
sentence_level_df = sentence_level_df_sorted.drop_duplicates(subset='text', keep='first')


In [16]:
from sklearn.model_selection import train_test_split
# Split train-test
train_df, test_df = train_test_split(
    sentence_level_df,
    test_size=0.2,
    stratify=sentence_level_df['type'],
    random_state=42
)

feature_cols = [
    "text",
    "is_dialogue",
    "count_lexicon",
    "is_subject",
    "is_object"
]

target_col = "type"

X_train_exploded = train_df[feature_cols].copy()
y_train_exploded = train_df[target_col].copy()

X_test_exploded = test_df[feature_cols].copy()
y_test_exploded = test_df[target_col].copy()

In [17]:
train_df['type'].value_counts()

,count
type,
Protagonist,1665
Antagonist,1619


In [18]:
import pandas as pd
from collections import Counter
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
# Function to get the most common element from a list
def get_most_common_label(label_list):
    # Flatten the list if it contains nested lists
    # flat_list = [item for sublist in label_list for item in sublist]
    most_common = Counter(label_list).most_common(1)[0][0]
    return most_common

def evaluation_metric_val(best_model, val_df):
    data_test = val_df.copy()
    predictions = best_model.predict(data_test)
    data_test['predicted_type'] = predictions

    # Join alias lists into a string
    data_test["aliases"] = data_test["aliases"].apply(lambda x: ", ".join(x) if isinstance(x, list) else x)

    # Group by entity and aggregate predictions and true labels
    data_test = data_test.groupby(["aliases", "story_id", "person"])[["predicted_type", "type"]].agg(list).reset_index()

    # Get most common label per group
    data_test['predicted_type_group'] = data_test['predicted_type'].apply(get_most_common_label)
    data_test['type_grouped'] = data_test['type'].apply(get_most_common_label)

    # Ground truth and predictions
    y_true = data_test['type_grouped']
    y_pred = data_test['predicted_type_group']

    # Compute confusion matrix
    cm = confusion_matrix(y_true, y_pred)

    # Display confusion matrix
    print("Confusion Matrix:\n", cm)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm)
    disp.plot(cmap='Blues')

    # Print classification report
    print("Classification Report:")
    print(classification_report(y_true, y_pred, digits=4))

    # Compute and print individual metrics
    precision = precision_score(y_true, y_pred, average='macro')
    recall = recall_score(y_true, y_pred, average='macro')
    f1 = f1_score(y_true, y_pred, average='macro')
    accuracy = accuracy_score(y_true, y_pred)

    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1-score:  {f1:.4f}")
    print(f"Accuracy:  {accuracy:.4f}")
    return data_test


In [19]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MinMaxScaler
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report

import os
import joblib
from google.colab import drive

drive.mount('/content/drive')

save_dir = "/content/drive/MyDrive/ML_CharacterType/ferro_svm_bali"
os.makedirs(save_dir, exist_ok=True)

# Preprocessing pipeline
# Preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('integer', MinMaxScaler(), ["is_dialogue", "is_subject", "count_lexicon"]),
        ('text', TfidfVectorizer(), 'text')
    ]
)
# Pipeline with LinearSVC
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LinearSVC())
])

# Grid search parameters
param_grid = {
    'preprocessor__text__max_df': [0.75, 1.0],
    'preprocessor__text__ngram_range': [(1, 1), (1, 2)],
    'preprocessor__text__max_features': [100, 500, 1000, 1500, 2000, 2500, 3000, 5000],
    'classifier__C': [0.1, 1, 10],  # Regularization parameter for SVM
    'classifier__loss': ['hinge', 'squared_hinge'],  # SVM loss functions
    'classifier__max_iter': [1000, 5000],  # max iterations
    'classifier__class_weight': [None, 'balanced']
}

# GridSearchCV
grid_search = GridSearchCV(pipeline, param_grid, cv=5, n_jobs=-1, verbose=1)

# Fit grid search
grid_search.fit(X_train_exploded, y_train_exploded)

# Best parameters and score
print("Best parameters:", grid_search.best_params_)
print("Best cross-validation score:", grid_search.best_score_)

# Evaluate on test set
test_score = grid_search.score(X_test_exploded, y_test_exploded)
y_pred = grid_search.best_estimator_.predict(X_test_exploded)

print("Test set score:", test_score)
report = classification_report(y_test_exploded, y_pred)
print("Classification Report for Best Model:\n", report)

best_model_one_svm_exp_tf = grid_search.best_estimator_
train_accuracy = best_model_one_svm_exp_tf.score(X_train_exploded, y_train_exploded)
test_accuracy = best_model_one_svm_exp_tf.score(X_test_exploded, y_test_exploded)

print("Train Accuracy:", train_accuracy)
print("Test Accuracy:", test_accuracy)

joblib.dump(best_model_one_svm_exp_tf, os.path.join(save_dir, "best_model.pkl"))

with open(os.path.join(save_dir, "best_params.txt"), "w") as f:
    f.write("Best parameters:\n")
    f.write(str(grid_search.best_params_))
    f.write("\n\nBest cross-validation score:\n")
    f.write(str(grid_search.best_score_))
    f.write("\n\nTrain Accuracy:\n")
    f.write(str(train_accuracy))
    f.write("\n\nTest Accuracy:\n")
    f.write(str(test_accuracy))

with open(os.path.join(save_dir, "classification_report.txt"), "w") as f:
    f.write(report)

pred_df = X_test_exploded.copy()
pred_df["y_true"] = y_test_exploded.values if hasattr(y_test_exploded, "values") else y_test_exploded
pred_df["y_pred"] = y_pred
pred_df.to_csv(os.path.join(save_dir, "test_predictions.csv"), index=False)

print(f"Semua hasil berhasil disimpan di: {save_dir}")


Mounted at /content/drive
Fitting 5 folds for each of 768 candidates, totalling 3840 fits
Best parameters: {'classifier__C': 1, 'classifier__class_weight': None, 'classifier__loss': 'hinge', 'classifier__max_iter': 1000, 'preprocessor__text__max_df': 0.75, 'preprocessor__text__max_features': 5000, 'preprocessor__text__ngram_range': (1, 2)}
Best cross-validation score: 0.793842113078665
Test set score: 0.8099878197320342
Classification Report for Best Model:
               precision    recall  f1-score   support

  Antagonist       0.79      0.83      0.81       405
 Protagonist       0.83      0.79      0.81       416

    accuracy                           0.81       821
   macro avg       0.81      0.81      0.81       821
weighted avg       0.81      0.81      0.81       821

Train Accuracy: 0.942448233861145
Test Accuracy: 0.8099878197320342
Semua hasil berhasil disimpan di: /content/drive/MyDrive/ML_CharacterType/ferro_svm_bali
